# VAR: 다중 시계열 예측

SARIMAX는 외생 변수가 시계열에 미치는 영향. 관계가 **단방향**이므로 외생 변수가 대상에만 영향을 미친다고 가정.  

하지만 두 시계열이 양방향 관계를 가질수도 있음. 즉, 시계열 t1이 시계열 t2의 예측 변수이고, 시계열 t2가 시계열 t1의 예측 변수가 될 수도 있음. 이런 경우 이 양방향 관계를 고려해 두 시계열에 대한 예측을 동시에 출력할 수 있는 모델이 있다면 유용.  

VAR: Vector Autoregression (벡터자기회귀)
- 시간에 따라 변화하는 여러 시계열 간의 관계를 포착
- 여러 시계열에 대한 예측을 동시에 생성할 수 있어 다변량 예측(multivariate forecasting)을 수행

Ch 9와 똑같은 가처분 소득과 실질 소비 사이의 관계를 살펴봄. 

In [1]:
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np

macro_econ_data = sm.datasets.macrodata.load_pandas().data
macro_econ_data

# realcons: 실질 소비
# realdpi: 가처분 소득


,year,quarter,realgdp,realcons,realinv,realgovt,realdpi,cpi,m1,tbilrate,unemp,pop,infl,realint
0,1959.0,1.0,2710.349,1707.4,286.898,470.045,1886.9,28.980,139.7,2.82,5.8,177.146,0.00,0.00
1,1959.0,2.0,2778.801,1733.7,310.859,481.301,1919.7,29.150,141.7,3.08,5.1,177.830,2.34,0.74
2,1959.0,3.0,2775.488,1751.8,289.226,491.260,1916.4,29.350,140.5,3.82,5.3,178.657,2.74,1.09
3,1959.0,4.0,2785.204,1753.7,299.356,484.052,1931.3,29.370,140.0,4.33,5.6,179.386,0.27,4.06
4,1960.0,1.0,2847.699,1770.5,331.722,462.199,1955.5,29.540,139.6,3.50,5.2,180.007,2.31,1.19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198,2008.0,3.0,13324.600,9267.7,1990.693,991.551,9838.3,216.889,1474.7,1.17,6.0,305.270,-3.16,4.33
199,2008.0,4.0,13141.920,9195.3,1857.661,1007.273,9920.4,212.174,1576.5,0.12,6.9,305.952,-8.79,8.91
200,2009.0,1.0,12925.410,9209.2,1558.494,996.287,9926.4,212.671,1592.8,0.22,8.1,306.547,0.94,-0.71
201,2009.0,2.0,12901.504,9189.0,1456.678,1023.528,10077.5,214.469,1653.6,0.18,9.2,307.226,3.37,-3.19


즉, 가처분 소득이 많을수록 소비가 많다는 가설 설립.  
반대로 소비가 많다는 것은 더 많은 소득을 소비할 수 있다는 뜻일수도.  
이런 양방향 관계를 VAR 모델로 포착.

# 10.1 VAR 모델 살펴보기

VAR 모델: 시간에 따라 변화하는 여러 수열 간의 관계를 포착.
- VAR에서는 각 수열이 다른 수열에 영향을 미치지만, 
- SARIMAX 모델에서는 이와 달리 외생 변수는 대상에 영향을 미쳐도 그 반대는 성립하지 않음. (즉, 대상은 외생 변수에 영향을 미친다 생각하지 않음?)
    - Ch 9에서 다른 변수들은 realgdp의 예측 변수로 사용했음. realgdp가 이런 변수들에 어떤 영향을 미쳤는지는 고려하지 않음. 그래서 이 때는 SARIMAX 모델을 사용.

---
이제, 다시 자기회귀로 돌아옴. 
- VAR 모델은 여러 시계열을 예측할 수 있도록 AR(p) 모델을 일반화한 것.
    - VAR 모델을 VAR(p)로 나타낼 수도 있음. 여기서 p는 차수이고, AR(p) 모델과 동일한 의미.

$y_t = C + \phi_1y_{t-1}+\phi_2y_{t-2}+...+\phi_py_{t-p}+\epsilon_t$  
위 식을 확장해 각각의 시계열이 서로에게 영향을 미치는 모델을 만들 수 있음.  
$y_{1,t}$ 와 $y_{2,t}$ 로 표시하는 두 개의 시계열이 있고 차수가 1 (p = 1)인 시스템을 고려해보자. 

$\begin{bmatrix} y_{1,t} \\ y_{2,t} \end{bmatrix} = \begin{bmatrix} C_1 \\ C_2 \end{bmatrix} + \begin{bmatrix} \phi_{1,1} & \phi_{1,2} \\ \phi_{2,1} & \phi_{2,2} \end{bmatrix} \begin{bmatrix} y_{1, t-1} \\ y_{2, t-1} \end{bmatrix} + \begin{bmatrix} \epsilon_{1,t} \\ \epsilon_{2,t} \end{bmatrix} $

- 좌변의 $y_{1,t}, y_{2,t}$: 시각 t에서의 두 시계열 변수의 값. 예를 들어 $y_{1,t}$ 가 GDP이고, $y_{2,t}$ 가 금리라면, "이번 기 GDP"와 "이번 기 금리"를 나타냄.
- 상수 $C_{1}​,C_{2}$: 방정식의 intercept. AR 모델에서의 상수항과 같은 역할, 시계열의 평균 수준을 결정. $C_1$은 $y_1$ 방정식의 상수, $C_2$는 $y_2$ 방정식의 상수
- 계수 $\phi_{1,1}, \phi_{1,2}, \phi_{2,1}, \phi_{2,2}$
    - **이게 VAR 모델의 핵심**
    - $\phi_{1,1}$: $y_{1,t-1}$ 이 $y_{1,t}$ 에 미치는 영향 (자기 자신의 과거 --> 자기 자신 (AR 부분))
    - $\phi_{1,2}$: $y_{2,t-1}$ 이 $y_{1,t}$ 에 미치는 영향 (상대방의 과거 --> 자기 자신, 교차 효과)
    - $\phi_{2,1}$: $y_{1,t-1}$ 이 $y_{2,t}$ 에 미치는 영향 (교차 효과)
    - $\phi_{2,2}$: $y_{2,t-1}$ 이 $y_{2,t}$ 에 미치는 영향 (AR 부분)
        - ($\phi_{1,1}$, $\phi_{2,2}$) 는 '자기 시차 효과'를, ($\phi_{1,2}$, $\phi_{2,1}$)는 '상호 시차 효과'를 담당.
            - ($\phi_{1,2}$, $\phi_{2,1}$)가 단변량 AR과 VAR을 구분 짓는 지점. $y_2$의 과거가 $y_1$의 현재를 설명할 수 있다는 것이 VAR. 
- 오차항 $\epsilon_{1,t}, \epsilon_{2,t}$: 각 방정식의 white noise 오차. VAR에서는 이 둘이 서로 상관관계를 가질 수 있다고 가정. 즉 $Cov(\epsilon_{1,t}, \epsilon_{2,t}) \neq 0$ 일 수 있음.

행렬 곱셈을 수행하면 $y_{1,t}$ 에 대한 수식과 $y_{2,t}$에 대한 수식:
- $y_{1,t} = C_1+\phi_{1,1}y_{1,t-1}+\phi_{1,2}y_{2,t-1}+\epsilon_{1,t}$
    - $y_{1,t}$ 의 식에 $y_{2,t}$ 의 과것값이 포함.
- $y_{2,t} = C_2+\phi_{2,1}y_{1,t-1}+\phi_{2,2}y_{2,t-1}+\epsilon_{2,t}$
    - $y_{2,t}$ 의 식에 $y_{1,t}$ 의 과것값이 포함.

